# RoadScan v2 — YOLO detection experiment

This experiment starts from COCO-pretrained YOLO11s and uses exactly the 8,999 training images used by the ResNet-18 v2 baseline. Japan, India and Czech are training countries; China (1,977 images) is the validation domain. The United States (4,805 images) and local photographs are reserved for a separate final evaluation after the recipe and thresholds are fixed. No weights, predictions or processed labels from notebooks 04–13 are reused.

Detection mAP and image-level Macro-F1 answer different questions and are reported separately. Baseline reference: China Macro-F1 0.496, US Macro-F1 0.362. China results are development results, because checkpoint selection and threshold calibration use this set. No target score or absence of overfitting is guaranteed.

## 1. Validate and stage the fixed sample

The following cell checks image–label pairing, manifest equality, country counts, normalized YOLO boxes and exact image duplicates between training and validation. It copies data to local working storage for predictable training access. Labels remain unchanged. This audit does not exclude near-duplicate scenes; that remains a limitation.

In [1]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib, json, shutil, time
import numpy as np
from PIL import Image
import yaml

BASE = Path('/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2')
SOURCE = BASE / 'sample9000'
ROOT = Path('/kaggle/working/sample9000')
ART = Path('/kaggle/working/artifacts_v2')
ART.mkdir(exist_ok=True)
CLASSES = ['D00', 'D10', 'D20', 'D40']
manifest = (BASE / 'sample9000_list.txt').read_text().splitlines()
assert len(manifest) == len(set(manifest)) == 8999
jobs = []
for split, expected in [('train', 8999), ('val', 1977)]:
    images = {p.stem: p for p in (SOURCE/split/'images').glob('*.jpg')}
    labels = {p.stem: p for p in (SOURCE/split/'labels').glob('*.txt')}
    assert images.keys() == labels.keys() and len(images) == expected
    if split == 'train':
        assert set(images) == set(manifest)
        assert Counter(s.split('_')[0] for s in images) == {'Japan':4000, 'India':3500, 'Czech':1499}
    for folder in ['images', 'labels']:
        (ROOT/split/folder).mkdir(parents=True, exist_ok=True)
    jobs += [(split, s, images[s], labels[s]) for s in sorted(images)]

def inspect_and_copy(job):
    split, stem, image, label = job
    raw = label.read_text()
    rows = [list(map(float, line.split())) for line in raw.splitlines() if line.strip()]
    for row in rows:
        assert len(row) == 5 and np.isfinite(row).all(), (split, stem, row)
        cls, x, y, w, h = row
        assert cls == int(cls) and 0 <= cls < 4
        assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
        assert x-w/2 >= -1e-5 and y-h/2 >= -1e-5 and x+w/2 <= 1.00001 and y+h/2 <= 1.00001
    data = image.read_bytes()
    target = ROOT/split/'images'/image.name
    target.write_bytes(data)
    with Image.open(target) as im:
        im.verify()
    (ROOT/split/'labels'/label.name).write_text(raw)
    return {'split':split, 'stem':stem, 'sha256':hashlib.sha256(data).hexdigest(), 'boxes':len(rows), 'classes':sorted({int(r[0]) for r in rows})}

started = time.monotonic()
records, errors = [], []
with ThreadPoolExecutor(max_workers=24) as pool:
    futures = {pool.submit(inspect_and_copy,j):j[:2] for j in jobs}
    for f in as_completed(futures):
        try:
            records.append(f.result())
        except Exception as e:
            errors.append({'image':futures[f], 'error':repr(e)})
        if (len(records)+len(errors)) % 500 == 0:
            print('Audited:', len(records)+len(errors), '/', len(jobs), flush=True)
(ART/'audit.json').write_text(json.dumps({'records':records, 'errors':errors}, indent=2))
assert not errors, 'Audit failed: see artifacts_v2/audit.json'
train_hash = {r['sha256'] for r in records if r['split']=='train'}
val_hash = {r['sha256'] for r in records if r['split']=='val'}
assert not train_hash & val_hash, 'Exact duplicate images cross train/validation. Resolve with team before training.'
shutil.copy2(BASE/'sample9000_list.txt', ART/'sample9000_list.txt')
DATA = Path('/kaggle/working/data_v2.yaml')
DATA.write_text(yaml.safe_dump({'path':str(ROOT),'train':'train/images','val':'val/images','names':dict(enumerate(CLASSES))}, sort_keys=False))
for split in ['train','val']:
    subset = [r for r in records if r['split']==split]
    print(split, 'images:',len(subset), 'boxes:',sum(r['boxes'] for r in subset), 'empty:',sum(r['boxes']==0 for r in subset))
print('Audit passed. Minutes:', round((time.monotonic()-started)/60,1), flush=True)


Audited: 500 / 10976
Audited: 1000 / 10976
Audited: 1500 / 10976
Audited: 2000 / 10976
Audited: 2500 / 10976
Audited: 3000 / 10976
Audited: 3500 / 10976
Audited: 4000 / 10976
Audited: 4500 / 10976
Audited: 5000 / 10976
Audited: 5500 / 10976
Audited: 6000 / 10976
Audited: 6500 / 10976
Audited: 7000 / 10976
Audited: 7500 / 10976
Audited: 8000 / 10976
Audited: 8500 / 10976
Audited: 9000 / 10976
Audited: 9500 / 10976
Audited: 10000 / 10976
Audited: 10500 / 10976
train images: 8999 boxes: 18354 empty: 1427
val images: 1977 boxes: 4650 empty: 43
Audit passed. Minutes: 5.5


## 2. Fine-tune a COCO-pretrained detector

YOLO11s at 640 pixels provides a practical first run before testing larger resolutions. All trainable layers are fine-tuned using AdamW, cosine learning-rate decay, three warmup epochs and weight decay. Mosaic, horizontal flips, scale and HSV changes are applied to training images only. Small rotations are avoided because D00 and D10 depend on crack direction.

The run allows 40 epochs with early stopping after 8 epochs without validation improvement. A callback requests a stop at the next completed epoch once 90 minutes have elapsed; validation and artifact export add time. Epoch scheduling remains fixed. The checkpoint with the best validation detection fitness is selected. GPU availability is required. PyTorch is reused from Kaggle rather than reinstalled.

References: https://docs.ultralytics.com/models/yolo11/ and https://docs.ultralytics.com/modes/train/

In [2]:
import sys, subprocess, importlib.metadata, platform
try:
    importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert DATA.exists() and len(records) == 10976 and not errors
assert torch.cuda.is_available(), 'Select a GPU accelerator before running training.'
versions = {'python':platform.python_version(), 'torch':torch.__version__, 'ultralytics':ultralytics.__version__, 'gpu':torch.cuda.get_device_name(0)}
(ART/'environment.json').write_text(json.dumps(versions, indent=2))
print(versions)
model = YOLO('yolo11s.pt')
clock = {}
def start_clock(trainer):
    clock['start'] = time.monotonic()
def budget_stop(trainer):
    if time.monotonic() - clock['start'] >= 90*60:
        trainer.stop = True
        print('90-minute training budget reached; finishing this epoch and saving.')
model.add_callback('on_train_start', start_clock)
model.add_callback('on_fit_epoch_end', budget_stop)
train_results = model.train(
    data=str(DATA), imgsz=640, epochs=40, patience=8, batch=16,
    device=0, workers=2, seed=42, deterministic=True,
    optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True,
    warmup_epochs=3, weight_decay=0.0005,
    mosaic=1.0, close_mosaic=10, mixup=0.0,
    scale=0.5, translate=0.1, degrees=0.0, fliplr=0.5, flipud=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    cache=False, amp=True, save=True, save_period=5, plots=True,
    project='/kaggle/working/runs_v2', name='yolo11s_640_seed42',
)
RUN = Path(model.trainer.save_dir)
BEST = RUN/'weights/best.pt'
assert BEST.exists()
shutil.copy2(BEST, ART/'yolo_v2_best.pt')
shutil.copy2(RUN/'results.csv', ART/'training_history.csv')
shutil.copy2(RUN/'args.yaml', ART/'training_args.yaml')
print('Best model saved:', ART/'yolo_v2_best.pt')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 4.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{'python': '3.13.15', 'torch': '2.11.0+cu128', 'ultralytics': '8.4.177', 'gpu': 'Tesla T4'}
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosai

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/40      5.02G      2.214      2.684       1.96         30        640: 100% ━━━━━━━━━━━━ 563/563 5.2it/s 1:49
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.0it/s 15.5s
                   all       1977       4650     0.0878       0.15     0.0487     0.0144

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/40      5.02G      2.188      2.635      1.945         23        640: 100% ━━━━━━━━━━━━ 563/563 5.2it/s 1:48
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.3it/s 14.3s
                   all       1977       4650      0.102      0.147     0.0402      0.011

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/40      5.02G      2.139      2.543      1.899         20        640: 100% ━━━━━━━━━━━━ 563/563 5.2it/s 1:48
                 Class     Images  Instances      Box(

## 3. Evaluate detection and calibrate image-level decisions

Detection is evaluated on the complete China validation set using standard mAP50 and mAP50–95. For the ResNet comparison, each image receives four scores: the maximum retained detection confidence for each class, or zero when no detection is present. Predictions are retained down to confidence 0.001; this inference floor is fixed before evaluation.

Class thresholds are selected on China from 0.05 to 0.95 in steps of 0.05, matching the baseline grid. Both fixed-threshold (0.5) and calibrated Macro-F1 are saved, together with AUROC, per-class precision/recall/F1, and the always-positive reference. Calibrated validation F1 is optimistic and is not an independent test result. The saved thresholds must be reused unchanged on the United States and local photographs. Training history and country-specific training diagnostics will inform follow-up work; this run alone cannot establish generalization to every city.

In [3]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
best_model = YOLO(str(ART/'yolo_v2_best.pt'))
detection = best_model.val(data=str(DATA), split='val', imgsz=640, batch=16, device=0, workers=2, plots=True, project='/kaggle/working/evaluation_v2', name='china')
(ART/'china_detection.json').write_text(json.dumps({str(k):float(v) for k,v in detection.results_dict.items()}, indent=2))
stems = sorted(p.stem for p in (ROOT/'val/labels').glob('*.txt'))
truth = np.zeros((len(stems),4), dtype=int)
scores = np.zeros((len(stems),4), dtype=float)
lookup = {stem:i for i,stem in enumerate(stems)}
for i,stem in enumerate(stems):
    for line in (ROOT/'val/labels'/f'{stem}.txt').read_text().splitlines():
        if line.strip():
            truth[i,int(line.split()[0])] = 1
for result in best_model.predict(source=str(ROOT/'val/images'), imgsz=640, conf=0.001, iou=0.7, max_det=300, device=0, stream=True, verbose=False):
    i = lookup[Path(result.path).stem]
    for cls, conf in zip(result.boxes.cls.cpu().numpy().astype(int), result.boxes.conf.cpu().numpy()):
        scores[i,cls] = max(scores[i,cls], float(conf))
grid = np.arange(0.05,0.96,0.05)
thresholds = []
for c in range(4):
    values = [precision_recall_fscore_support(truth[:,c], scores[:,c]>=t, average='binary', zero_division=0)[2] for t in grid]
    thresholds.append(float(grid[int(np.argmax(values))]))
report = []
for mode, threshold in [('fixed_0.5', np.full(4,0.5)),('calibrated',np.array(thresholds))]:
    p,r,f,_ = precision_recall_fscore_support(truth,scores>=threshold,zero_division=0)
    for c,name in enumerate(CLASSES):
        prevalence = truth[:,c].mean()
        auc = roc_auc_score(truth[:,c],scores[:,c]) if len(np.unique(truth[:,c]))==2 else None
        report.append({'domain':'China','mode':mode,'class':name,'threshold':float(threshold[c]),'precision':float(p[c]),'recall':float(r[c]),'f1':float(f[c]),'auroc':auc,'always_positive_f1':float(2*prevalence/(1+prevalence))})
    print(mode,'Macro-F1:',round(float(f.mean()),4))
pd.DataFrame(report).to_csv(ART/'china_image_metrics.csv',index=False)
np.savez_compressed(ART/'china_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
(ART/'inference_config.json').write_text(json.dumps({'classes':CLASSES,'thresholds':thresholds,'imgsz':640,'conf_floor':0.001,'iou':0.7,'max_det':300,'seed':42,'threshold_source':'China validation only'},indent=2))
print(pd.DataFrame(report).to_string(index=False))
print('Saved artifacts:',sorted(p.name for p in ART.iterdir()))
print('Next: freeze recipe and evaluate the US and local test images. No test scores were used here.')


Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1847.7±570.0 MB/s, size: 79.0 KB)
val: Scanning /kaggle/working/sample9000/val/labels.cache... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 518.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 4.0it/s 30.8s
                   all       1977       4650      0.341      0.302      0.222     0.0868
                   D00       1419       2678      0.321      0.149      0.134     0.0488
                   D10        738       1096      0.357      0.282       0.25     0.0718
                   D20        486        641      0.547      0.402      0.398      0.194
                   D40        164        235      0.139      0.374      0.107     0.0329
Speed: 1.1ms preprocess, 10.